In [14]:
import sys
import os

sys.path.append(os.getcwd())

In [15]:
import cv2
import numpy as np
import timeit

from common_lab_utils import Circle, CircleEstimate,\
    create_1d_gaussian_kernel, extract_inlier_points, retain_best, \
    draw_corner_result

def create_1d_derivated_gaussian_kernel(sigma, radius=0):
    """
    Creates a Nx1 derivated gaussian filter kernel.

    :param sigma: The sigma (standard deviation) parameter for the gaussian.
    :param radius: The filter radius, so that N = 2*radius + 1. If set to 0, the radius will be computed so that radius = 3.5 * sigma.
    :return:
    """
    if radius <= 0:
        radius = int(np.ceil(3.5 * sigma))

    # TODO 1: Use create1DGaussianKernel to compute the derivated kernel.
    kernel, x = create_1d_gaussian_kernel(sigma, radius)

    kernel = kernel * x * (-1. / (sigma * sigma))

    return kernel, x


class CornerDetector:
    """A homebrewed corner detector!"""

    def __init__(self, metric_type, visualize=False, quality_level=0.01, gradient_sigma=1.0, window_sigma=2.0):
        """
        Constructs the corner detector.

        :param metric_type: The metric used to extract corners.
        :param visualize: Shows additional debug/educational figures when true.
        :param quality_level: The quality level used for thresholding areas with corners.
        :param gradient_sigma: The standard deviation for the gradient filter.
        :param window_sigma: The standard deviation for the window filters.
        """

        self._metric_type = metric_type
        self._visualize = visualize
        self._quality_level = quality_level
        self._window_sigma = window_sigma
        self._g_kernel = create_1d_gaussian_kernel(gradient_sigma)[0]
        self._dg_kernel = create_1d_derivated_gaussian_kernel(gradient_sigma)[0]
        self._win_kernel = create_1d_gaussian_kernel(window_sigma)[0]

    def detect(self, image):
        """
        Detects corners in an image.

        :param image: The image that is queried for corners.
        :return: An array of corner key points.
        """

        # TODO 2: Estimate image gradients Ix and Iy by combining _g_kernel and _dg_kernel.
        ix = cv2.sepFilter2D(image, cv2.CV_32F, self._dg_kernel, self._g_kernel)
        iy = cv2.sepFilter2D(image, cv2.CV_32F, self._g_kernel, self._dg_kernel)

        # TODO 3.1: Compute the elements of M; A, B and C from Ix and Iy.
        a = ix * ix
        b = ix * iy
        c = iy * iy

        # TODO 3.2: Apply the windowing gaussian win_kernel_ on A, B and C.
        a = cv2.sepFilter2D(a, -1, self._win_kernel, self._win_kernel)
        b = cv2.sepFilter2D(b, -1, self._win_kernel, self._win_kernel)
        c = cv2.sepFilter2D(c, -1, self._win_kernel, self._win_kernel)

        # TODO 4: Finish all the corner response functions (see below).
        if self._metric_type == 'harris':
            response = self._harris_metric(a, b, c)
        elif self._metric_type == 'harmonic_mean':
            response = self._harmonic_metric(a, b, c)
        elif self._metric_type == 'min_eigen':
            response = self._min_eigen_metric(a, b, c)
        else:
            raise ValueError("metric_type must be 'harris', 'harmonic_mean' or 'min_eigen'")

        # TODO 5: Dilate image to make each pixel equal to the maximum in the neighborhood.
        local_max = cv2.dilate(response, np.ones((3, 3)))

        # TODO 6: Compute the threshold.
        threshold = response.max() * self._quality_level

        # TODO 7. Extract local maxima above threshold (response > threshold and response == local_max).
        is_strong_and_local_max = (response > threshold) & (response == local_max)
        max_locations = np.transpose(np.nonzero(is_strong_and_local_max))

        keypoint_size = 3.0 * self._window_sigma
        keypoints = np.array([cv2.KeyPoint(float(col), float(row), keypoint_size, -1, response[row, col]) for row, col in max_locations])

        if self._visualize:
            cv2.imshow("Gradient Ix", ix/25)
            cv2.imshow("Gradient Iy", iy/25)
            cv2.imshow("Gradient magnitude", (np.abs(ix) + np.abs(iy))/25)
            cv2.imshow("Image A", a)
            cv2.imshow("Image B", b)
            cv2.imshow("Image C", c)
            cv2.imshow("Response", response / (0.01 * response.max()))
            cv2.imshow("Local max", is_strong_and_local_max.astype(np.uint8) * 255)

        return keypoints, np.asarray(max_locations)

    @staticmethod
    def _harris_metric(a, b, c):
        # TODO 4.1: Finish the Harris metric.
        # Compute the Harris metric for each pixel.
        alpha = 0.06
        det_m = a * c - b * b
        trc_m = a + c

        return det_m - alpha * trc_m * trc_m

    @staticmethod
    def _harmonic_metric(a, b, c):
        # TODO 4.2 Finish the Harmonic Mean metric
        # Compute the Harmonic Mean metric for each pixel.
        det_m = a * c - b * b
        trc_m = a + c

        return det_m * 1./trc_m

    @staticmethod
    def _min_eigen_metric(a, b, c):
        # TODO 4.3 Finish the minimum eigenvalue metric
        # Compute the Min. Eigen metric for each pixel.
        root = np.sqrt(4. * np.square(b) + np.square(a-c))

        return 0.5 * ((a+c) - root)


class CircleEstimator:
    """A robust circle estimator based on circle point measurements"""

    def __init__(self, p=0.99, distance_threshold=5.0, max_iterations=np.iinfo(np.int32).max):
        """
        Constructs a circle estimator

        :param p: The desired probability of getting a good sample.
        :param distance_threshold: The maximum distance a good sample can have from the circle.
        :param max_iterations: The maximum number of iterations (set lower if slow)
        """
        self._p = p
        self._distance_threshold = distance_threshold
        self._max_iterations = max_iterations

    def estimate(self, points):
        """
        Estimates a circle based on the point measurements using RANSAC.

        :param points: Point measurements on the circle corrupted by noise.
        :return: The circle estimate based on the entire inlier set.
        """
        if points.shape[0] < 3:
            # Too few points to estimate any circle.
            return CircleEstimate()

        if len(points) == 3:
            # No need to estimate
            return CircleEstimate(circle=Circle.from_points(*points))

        # Estimate circle using RANSAC.
        ransac_estimate = self._ransac_estimator(points)

        # Check if valid result.
        if ransac_estimate.num_inliers < 3:
            return None

        # Extract inlier points.
        inlier_pts = extract_inlier_points(ransac_estimate, points)

        # Estimate circle based on all the inliers.
        refined_circle = self._least_squares_estimator(inlier_pts)

        return CircleEstimate(
            circle=refined_circle,
            num_iterations=ransac_estimate.num_iterations,
            num_inliers=ransac_estimate.num_inliers,
            is_inlier=ransac_estimate.is_inlier
        )

    def _ransac_estimator(self, points):
        """Perform RANSAC estimation"""

        # Initialize maximum number of iterations.
        num_iterations = self._max_iterations

        # Perform RANSAC
        iteration = 0
        best_circle = None
        best_num_inliers = 0
        best_is_inlier = np.array([], dtype=bool)

        while iteration < num_iterations:
            # Determine test circle by drawing minimal number of samples.
            test_circle = Circle.from_points(*points[np.random.choice(len(points), size=3, replace=False)])

            # Continue if the test circle was invalid.
            if not test_circle:
                continue

            # Count number of inliers.
            is_inlier = test_circle.distances(points) < self._distance_threshold
            test_num_inliers = np.count_nonzero(is_inlier)

            # Check if this estimate gave a better result.
            # TODO 8: Remove break and perform the correct test!
            if test_num_inliers > best_num_inliers:
                # Update circle with largest inlier set.
                best_num_inliers = test_num_inliers
                best_is_inlier = is_inlier
                best_circle = test_circle

                # Adaptively update number of iterations.
                inlier_ratio = best_num_inliers / len(points)
                if inlier_ratio == 1.0:
                    break

                num_iterations = np.minimum(
                    int(np.log(1.0 - self._p) / np.log(1.0 - inlier_ratio**3)),
                    self._max_iterations)

            iteration += 1

        return CircleEstimate(
            circle=best_circle,
            num_iterations=iteration,
            num_inliers=best_num_inliers,
            is_inlier=best_is_inlier
        )

    @staticmethod
    def _least_squares_estimator(points):
        """ Estimates the least squares solution for the parameters of a circle given the points.

        The equations for the points (x_i, y_i) on the circle (x_c, y_c, r) is:
            (x_i - x_c)^2 + (y_i - y_c)^2 = r^2

        By multiplying out, we get the linear equations
            (2*x_c)*x_i + (2*y_c)*y_i + (r^2 - x_c^2 - y_x^2) = x_i^2 + y_i^2

        The least-squares problem then has the form A*p = b, where
            A = [x_i, y_i, 1],
            p = [2*x_c, 2*y_c, r^2 - x_c^2 - y_x^2]^T,
            b = [x_i^2 + y_i^2]

        by solving for p = [p_0, p_1, p_2], we get the following estimates for the circle parameters:
            x_c = 0.5 * p_0,
            y_c = 0.5 * p_1,
            r = sqrt(p_2 + x_c^2 + y_c^2)
        """
        # Construct A and b.
        A = np.c_[points, np.ones(len(points))]
        b = np.sum(np.square(points), axis=1)

        # Determine solution for p.
        p = np.linalg.lstsq(A, b, rcond=None)[0]

        # Extract center point and radius from the parameter vector p.
        center_point = 0.5 * p[:2]
        radius = np.sqrt(p[2] + np.sum(np.square(center_point)))

        return Circle(center_point, radius)

In [16]:
def run_corners_solution():
    # Connect to the camera.
    video_source = "examples/example.mp4"
    cap = cv2.VideoCapture(video_source)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    if not cap.isOpened():
        print(f"Could not open video source {video_source}")
        return
    else:
        print(f"Successfully opened video source {video_source}")

    # Create window
    window_name = 'Solution: Estimating circles from corners'
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    # Construct the corner detector.
    # Play around with the parameters!
    # When the second argument is true, additional debug visualizations are shown.
    det = CornerDetector(metric_type='harris', visualize=False)

    # Construct the circle estimator
    estimator = CircleEstimator()

    while True:
        # Read next frame.
        success, frame = cap.read()
        if not success:
            print(f"The video source {video_source} stopped")
            break

        # Convert frame to gray scale image.
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Perform corner detection
        # Measure how long the processing takes.
        start = timeit.default_timer()

        keypoints, points = det.detect(gray_frame)

        end = timeit.default_timer()
        duration_corners = end - start

        # Keep the highest scoring points.
        best = retain_best(keypoints, 1000)
        keypoints = keypoints[best]
        points = points[best]

        # Estimate circle based on detected corner points
        start = timeit.default_timer()

        circle_estimate = estimator.estimate(points)

        end = timeit.default_timer()
        duration_circle = end - start

        # Show the results
        draw_corner_result(frame, keypoints, duration_corners)
        # draw_circle_result(frame, keypoints, circle_estimate, duration_circle)
        cv2.imshow(window_name, frame)

        # Update the GUI and wait a short time for input from the keyboard.
        key = cv2.waitKey(1)

        # React to keyboard commands.
        if key == ord('q'):
            print("Quitting")
            break

    # Stop video source.
    cv2.destroyAllWindows()
    cap.release()

In [17]:
import numpy as np
import cv2 as cv
import argparse

# Check OpenCV version
opencv_python_version = lambda str_version: tuple(map(int, (str_version.split("."))))
assert opencv_python_version(cv.__version__) >= opencv_python_version("4.10.0"), \
       "Please install latest opencv-python for benchmark: python3 -m pip install --upgrade opencv-python"

from nanodet import NanoDet

# Valid combinations of backends and targets
backend_target_pairs = [
    [cv.dnn.DNN_BACKEND_OPENCV, cv.dnn.DNN_TARGET_CPU],
    [cv.dnn.DNN_BACKEND_CUDA,   cv.dnn.DNN_TARGET_CUDA],
    [cv.dnn.DNN_BACKEND_CUDA,   cv.dnn.DNN_TARGET_CUDA_FP16],
    [cv.dnn.DNN_BACKEND_TIMVX,  cv.dnn.DNN_TARGET_NPU],
    [cv.dnn.DNN_BACKEND_CANN,   cv.dnn.DNN_TARGET_NPU]
]

classes = ('person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
           'train', 'truck', 'boat', 'traffic light', 'fire hydrant',
           'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog',
           'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe',
           'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
           'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat',
           'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
           'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
           'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot',
           'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
           'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
           'mouse', 'remote', 'keyboard', 'cell phone', 'microwave',
           'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock',
           'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush')

def letterbox(srcimg, target_size=(416, 416)):
    img = srcimg.copy()

    top, left, newh, neww = 0, 0, target_size[0], target_size[1]
    if img.shape[0] != img.shape[1]:
        hw_scale = img.shape[0] / img.shape[1]
        if hw_scale > 1:
            newh, neww = target_size[0], int(target_size[1] / hw_scale)
            img = cv.resize(img, (neww, newh), interpolation=cv.INTER_AREA)
            left = int((target_size[1] - neww) * 0.5)
            img = cv.copyMakeBorder(img, 0, 0, left, target_size[1] - neww - left, cv.BORDER_CONSTANT, value=0)  # add border
        else:
            newh, neww = int(target_size[0] * hw_scale), target_size[1]
            img = cv.resize(img, (neww, newh), interpolation=cv.INTER_AREA)
            top = int((target_size[0] - newh) * 0.5)
            img = cv.copyMakeBorder(img, top, target_size[0] - newh - top, 0, 0, cv.BORDER_CONSTANT, value=0)
    else:
        img = cv.resize(img, target_size, interpolation=cv.INTER_AREA)

    letterbox_scale = [top, left, newh, neww]
    return img, letterbox_scale

def unletterbox(bbox, original_image_shape, letterbox_scale):
    ret = bbox.copy()

    h, w = original_image_shape
    top, left, newh, neww = letterbox_scale

    if h == w:
        ratio = h / newh
        ret = ret * ratio
        return ret

    ratioh, ratiow = h / newh, w / neww
    ret[0] = max((ret[0] - left) * ratiow, 0)
    ret[1] = max((ret[1] - top) * ratioh, 0)
    ret[2] = min((ret[2] - left) * ratiow, w)
    ret[3] = min((ret[3] - top) * ratioh, h)

    return ret.astype(np.int32)

def vis(preds, res_img, letterbox_scale, fps=None):
    ret = res_img.copy()

    # draw FPS
    if fps is not None:
        fps_label = "FPS: %.2f" % fps
        cv.putText(ret, fps_label, (10, 25), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # draw bboxes and labels
    for pred in preds:
        bbox = pred[:4]
        conf = pred[-2]
        classid = pred[-1].astype(np.int32)

        # bbox
        xmin, ymin, xmax, ymax = unletterbox(bbox, ret.shape[:2], letterbox_scale)
        cv.rectangle(ret, (xmin, ymin), (xmax, ymax), (0, 255, 0), thickness=2)

        # label
        label = "{:s}: {:.2f}".format(classes[classid], conf)
        cv.putText(ret, label, (xmin, ymin - 10), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), thickness=2)

    return ret

In [18]:
parser = argparse.ArgumentParser(description='Nanodet inference using OpenCV an contribution by Sri Siddarth Chakaravarthy part of GSOC_2022')
parser.add_argument('--input', '-i', type=str,
                    help='Path to the input image. Omit for using default camera.')
parser.add_argument('--model', '-m', type=str,
                    default='/Users/calllevels/Desktop/Exchange Courses/TEK5030/Project/Corder Detection Box/object_detection_nanodet_2022nov.onnx', help="Path to the model")
parser.add_argument('--backend_target', '-bt', type=int, default=0,
                help='''Choose one of the backend-target pair to run this demo:
                    {:d}: (default) OpenCV implementation + CPU,
                    {:d}: CUDA + GPU (CUDA),
                    {:d}: CUDA + GPU (CUDA FP16),
                    {:d}: TIM-VX + NPU,
                    {:d}: CANN + NPU
                '''.format(*[x for x in range(len(backend_target_pairs))]))
parser.add_argument('--confidence', default=0.35, type=float,
                    help='Class confidence')
parser.add_argument('--nms', default=0.6, type=float,
                    help='Enter nms IOU threshold')
parser.add_argument('--save', '-s', action='store_true',
                    help='Specify to save results. This flag is invalid when using camera.')
parser.add_argument('--vis', '-v', action='store_true',
                    help='Specify to open a window for result visualization. This flag is invalid when using camera.')
args = parser.parse_args(args=[])

backend_id = backend_target_pairs[args.backend_target][0]
target_id = backend_target_pairs[args.backend_target][1]

model = NanoDet(modelPath= args.model,
                prob_threshold=args.confidence,
                iou_threshold=args.nms,
                backend_id=backend_id,
                target_id=target_id)

tm = cv.TickMeter()
tm.reset()

video_source = "/Users/calllevels/Desktop/Exchange Courses/TEK5030/Project/examples/example.mp4"
cap = cv2.VideoCapture(video_source)

window_name = "NanoDet + Corners"

det = CornerDetector(metric_type='harris', visualize=False)

init_done = False
prev_gray = None
prev_points = None

duration_corners = 0

backSub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=False)

while True:
    success, frame = cap.read()

    if not success:
        print(f"The video source {video_source} stopped")
        break

    # -----------------------------
    # PERSON DETECTION
    # -----------------------------
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    input_blob, letterbox_scale = letterbox(rgb)

    preds = model.infer(input_blob)

    # -----------------------------
    # PROCESS EACH DETECTED PERSON
    # -----------------------------
    for pred in preds:

        classid = int(pred[-1])

        # COCO class 0 = person
        if classid != 0:
            continue

        bbox = pred[:4]

        xmin, ymin, xmax, ymax = unletterbox(
            bbox,
            frame.shape[:2],
            letterbox_scale
        )

        xmin = max(0, xmin)
        ymin = max(0, ymin)
        xmax = min(frame.shape[1], xmax)
        ymax = min(frame.shape[0], ymax)

        # Draw bounding box
        cv2.rectangle(
            frame,
            (xmin, ymin),
            (xmax, ymax),
            (0, 255, 0),
            2
        )

        # -----------------------------
        # CROP PERSON ROI
        # -----------------------------
        person_roi = frame[ymin:ymax, xmin:xmax]

        if person_roi.size == 0:
            continue

        gray_roi = cv2.cvtColor(person_roi, cv2.COLOR_BGR2GRAY)

        fg_mask = backSub.apply(gray_roi)

        # -----------------------------
        # CORNER DETECTION INSIDE ROI
        # -----------------------------
        start = timeit.default_timer()

        keypoints, points = det.detect(gray_roi)

        end = timeit.default_timer()
        duration_corners = end - start

        # Keep strongest corners
        best = retain_best(keypoints, 300)

        keypoints = keypoints[best]
        points = points[best]

        # -----------------------------
        # SHIFT KEYPOINTS BACK
        # TO FULL IMAGE COORDS
        # -----------------------------
        shifted_keypoints = []

        for kp in keypoints:

            shifted_kp = cv2.KeyPoint(
                kp.pt[0] + xmin,
                kp.pt[1] + ymin,
                kp.size,
                kp.angle,
                kp.response
            )

            shifted_keypoints.append(shifted_kp)

        # Draw corners
        draw_corner_result(
            frame,
            shifted_keypoints,
            duration_corners
        )

    cv2.imshow(window_name, frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        print("Quitting")
        cap.release()
        cv2.destroyAllWindows()
        break

Quitting


In [19]:
import os
print(os.getcwd())

/Users/calllevels/Desktop/Exchange Courses/TEK5030/Project/Corder Detection Box
